In [4]:
# !pip install -U duckduckgo-search

In [2]:
# !pip install openai-agents

### Setup

In [3]:
import os
# from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")
else:
    print("OPENAI_API_KEY is set.")


# client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

OPENAI_API_KEY is set.


### Step 1: Define the tools

In [4]:
@function_tool
def search_web(query: str):
    """Search the web using duckduckgo browser. Return 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"\u2705 Got results")
    return json.dumps(results, indent=2)

In [5]:
@function_tool
def fetch_url(url: str):
    """Fetching the content from url using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 Got text: {len(text)} characters")
            return text
    print(f"\u274c Failed to extract text from {url}.")
    return f"Could not extract text from {url}. Try a diffrent source."

### Step 2: System prompt

In [6]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 3: Define the Agent

In [7]:
research_agent = Agent(
    name="ResearchAgent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url],
)

### Lets have it!


In [8]:
result = await Runner.run(
    research_agent,
    input="Research the following topic and produce a comprehensive research brief: How AI is used in healthcare",
    max_turns=30,
)

✅ Got results
❌ Failed to extract text from https://mayomagazine.mayoclinic.org/2026/04/ai-in-healthcare/.
✅ Got text: 9805 characters
✅ Got text: 98719 characters


In [9]:
print(f"Agent: {result.last_agent.name}")
print(f"----")
display(Markdown(result.final_output))

Agent: ResearchAgent
----


I have gathered detailed information from two comprehensive sources:

1. Wikipedia article on Artificial Intelligence in Healthcare (https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare) - This source provides an extensive overview of how AI is used in different healthcare applications including disease diagnosis, electronic health records, clinical documentation, drug discovery, telehealth, treatment fields like cardiology, dermatology, oncology, radiology, and more. It highlights both the opportunities and the challenges such as ethical concerns, data privacy, algorithmic bias, regulation, and technological unemployment. It mentions key examples like AlphaFold for protein folding, AI in cancer detection, AI chatbots in mental health, and AI use in clinical decision support.
  
2. Cleveland Clinic Health Essentials article (https://health.clevelandclinic.org/ai-in-healthcare) - This source focuses on practical uses of AI in healthcare today, such as improving diagnostic imaging interpretation, speeding emergency response through image triage, personalizing treatment plans using data analysis, supporting research, and reducing administrative burdens for providers. It emphasizes that AI aids rather than replaces clinicians, and points to the potential of AI-powered personalized medicine and preventive care. The article also mentions ethical considerations like patient data privacy and cautions against relying on general large language models for medical advice.

Next, I will synthesize information from these sources along with any relevant insights derived from the data to prepare a comprehensive research brief on "How AI is used in healthcare."